In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_auc_score,
    roc_curve
)

In [35]:
import numpy as np
import pandas as pd

# =========================================
# CONFIG
# =========================================
N_ROWS = 5000                 # change if needed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# =========================================
# FEATURE GENERATION (REALISTIC RANGES)
# =========================================
age = np.random.randint(18, 95, N_ROWS)

gender = np.random.choice(["Male", "Female"], size=N_ROWS, p=[0.58, 0.42])

heart_rate = np.random.normal(80, 15, N_ROWS).clip(40, 180)

systolic_bp = np.random.normal(125, 20, N_ROWS).clip(80, 220)

diastolic_bp = np.random.normal(80, 12, N_ROWS).clip(40, 140)

oxygen_saturation = np.random.normal(96, 3, N_ROWS).clip(70, 100)

respiratory_rate = np.random.normal(18, 4, N_ROWS).clip(8, 40)

cholesterol = np.random.normal(190, 35, N_ROWS).clip(120, 350)

diabetes = np.random.choice([0, 1], size=N_ROWS, p=[0.78, 0.22])

hypertension = np.random.choice([0, 1], size=N_ROWS, p=[0.65, 0.35])

smoking = np.random.choice([0, 1], size=N_ROWS, p=[0.7, 0.3])

# =========================================
# CREATE DATAFRAME
# =========================================
df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "heart_rate": heart_rate,
    "systolic_bp": systolic_bp,
    "diastolic_bp": diastolic_bp,
    "oxygen_saturation": oxygen_saturation,
    "respiratory_rate": respiratory_rate,
    "cholesterol": cholesterol,
    "diabetes": diabetes,
    "hypertension": hypertension,
    "smoking": smoking
})

# =========================================
# MEDICAL RISK SCORE (LOGISTIC-FRIENDLY)
# =========================================
risk_score = (
    0.04 * age +
    0.03 * heart_rate -
    0.05 * systolic_bp -
    0.04 * diastolic_bp -
    0.9  * oxygen_saturation +
    0.25 * respiratory_rate +
    0.02 * cholesterol +
    1.2  * diabetes +
    1.4  * hypertension +
    1.0  * smoking
)

# =========================================
# SIGMOID → PROBABILITY
# =========================================
probability = 1 / (1 + np.exp(-risk_score / 50))

# =========================================
# PERFECTLY BALANCED TARGET
# =========================================
threshold = np.percentile(probability, 50)
df["cardiac_arrest"] = (probability >= threshold).astype(int)

# =========================================
# SHUFFLE & SAVE
# =========================================
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

df.to_csv("cardiac_arrest_balanced_dataset.csv", index=False)

# =========================================
# VALIDATION
# =========================================
print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["cardiac_arrest"].value_counts(normalize=True))

Dataset shape: (5000, 12)

Target distribution:
cardiac_arrest
0    0.5
1    0.5
Name: proportion, dtype: float64


In [193]:
dt.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [195]:
dt.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst'],
      dtype='object')

In [197]:
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [199]:
dt.describe()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [222]:
 X = dt.drop(columns=["diagnosis"])
y = dt["diagnosis"]

In [224]:
X = pd.get_dummies(X, drop_first=True)

In [226]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [228]:
X_train.shape

(455, 31)

In [230]:
X_test.head()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
120,865137,11.41,10.82,73.34,403.3,0.09373,0.06685,0.03512,0.02623,0.1667,...,12.82,15.97,83.74,510.5,0.1548,0.2390,0.21020,0.08958,0.3016,0.08523
250,884948,20.94,23.56,138.90,1364.0,0.10070,0.16060,0.27120,0.13100,0.2205,...,25.58,27.00,165.30,2010.0,0.1211,0.3172,0.69910,0.21050,0.3126,0.07849
375,901303,16.17,16.07,106.30,788.5,0.09880,0.14380,0.06651,0.05397,0.1990,...,16.97,19.14,113.10,861.5,0.1235,0.2550,0.21140,0.12510,0.3153,0.08960
99,862548,14.42,19.77,94.48,642.5,0.09752,0.11410,0.09388,0.05839,0.1879,...,16.33,30.86,109.50,826.4,0.1431,0.3026,0.31940,0.15650,0.2718,0.09353
455,9112085,13.38,30.72,86.34,557.2,0.09245,0.07426,0.02819,0.03264,0.1375,...,15.05,41.61,96.69,705.6,0.1172,0.1421,0.07003,0.07763,0.2196,0.07675


In [232]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [234]:
model = LogisticRegression()

model.fit(X_train_scaled, y_train)

LogisticRegression()

In [258]:
accuracy_score(y_test, y_pred)

0.9736842105263158